[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab3/lab3_autoencoder_mtat_v3.ipynb)

# Lab 3 — Autoencoder convolucional para MagnaTagATune (V3)

**Machine Learning for Sound and Music — UPF**

Este notebook adapta el ejercicio de autoencoders de la asignatura al conjunto MagnaTagATune (MTAT).

Se mantiene el flujo del notebook docente: preparación del conjunto, entrenamiento de un autoencoder baseline, mejora mediante una pérdida perceptual y comparación de las reconstrucciones.

## Experimentos

1. **Baseline:** arquitectura y entrenamiento del notebook de referencia.
2. **Mejorado:** más capas convolucionales y pérdida perceptual multiescala STFT.

La única adaptación para los audios largos es recortar un segmento fijo de un segundo. Ambos modelos reciben los mismos segmentos y conservan la compresión del baseline.

Los datos descargados, extraídos y preprocesados se guardan en Google Drive para reutilizarlos en otras sesiones. Los resultados de esta versión se guardan en `MyDrive/MLSM/Lab 3/results_v3`.

## 1. Dependencias

Se utiliza PyTorch para las redes y librosa para decodificar los MP3 y convertirlos a mono y 16 kHz. No reinstalamos PyTorch para evitar alterar la configuración CUDA de Colab.

In [17]:
# FFmpeg permite decodificar archivos MP3.
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Librerías auxiliares para audio, datos, gráficas y barras de progreso.
!pip -q install librosa==0.10.2.post1 scikit-learn matplotlib pandas tqdm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Importaciones y configuración

Los parámetros del experimento se concentran aquí. Modificar los parámetros de entrenamiento o la arquitectura no invalida la caché de audio. Solo cambiar los parámetros que afectan al preprocesamiento (frecuencia, duración, recorte o normalización) requiere preparar de nuevo los segmentos.

In [18]:
import os
import json
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from IPython.display import Audio, display

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Preprocesamiento de audio: coincide con el notebook docente.
SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 1
INPUT_LENGTH = SAMPLE_RATE * SEGMENT_SECONDS

# Compresión original del autoencoder de referencia.
BOTTLENECK_SIZE = 512
COMPRESSION_RATE = INPUT_LENGTH / BOTTLENECK_SIZE

# Parámetros de entrenamiento del experimento de referencia.
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
NUM_WORKERS = 0

print(f"Frecuencia de muestreo: {SAMPLE_RATE} Hz")
print(f"Longitud de entrada: {INPUT_LENGTH} muestras")
print(f"Bottleneck: {BOTTLENECK_SIZE}")
print(f"Compresión: {COMPRESSION_RATE:.2f}:1")

Dispositivo: cuda
GPU: Tesla T4
Frecuencia de muestreo: 16000 Hz
Longitud de entrada: 16000 muestras
Bottleneck: 512
Compresión: 31.25:1


## 3. Google Drive, descarga y extracción de MTAT

Se conserva el código de descarga y extracción. Los archivos se guardan en `MyDrive/MLSM/Lab 3/data`. Si ya están descargados y extraídos, se reutilizan.

Los resultados de V3 se guardan en una carpeta distinta para no sobrescribir las versiones anteriores.

In [19]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/MLSM/Lab 3/data")
RESULTS_DIR = Path("/content/drive/MyDrive/MLSM/Lab 3/results_v3")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_DIR = DATA_ROOT / "downloads"
AUDIO_DIR = DATA_ROOT / "audio"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos:", DATA_ROOT)
print("Carpeta de resultados:", RESULTS_DIR)

PARTS = ["mp3.zip.001", "mp3.zip.002", "mp3.zip.003"]
BASE_URL = "https://mirg.city.ac.uk/datasets/magnatagatune"

for part in PARTS:
    destination = DOWNLOAD_DIR / part
    if not destination.exists() or destination.stat().st_size == 0:
        print(f"Descargando {part}...")
    else:
        print(f"Archivo ya presente: {part}")
    subprocess.run(
        ["wget", "-c", f"{BASE_URL}/{part}", "-P", str(DOWNLOAD_DIR)],
        check=True
    )
    print(f"Tamaño local: {destination.stat().st_size / (1024 ** 3):.2f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta de datos: /content/drive/MyDrive/MLSM/Lab 3/data
Carpeta de resultados: /content/drive/MyDrive/MLSM/Lab 3/results_v3
Archivo ya presente: mp3.zip.001
Tamaño local: 1.02 GB
Archivo ya presente: mp3.zip.002
Tamaño local: 1.02 GB
Archivo ya presente: mp3.zip.003
Tamaño local: 0.72 GB


In [20]:
MERGED_ARCHIVE = DOWNLOAD_DIR / "mp3.zip"
expected_archive_size = sum((DOWNLOAD_DIR / part).stat().st_size for part in PARTS)

if not MERGED_ARCHIVE.exists() or MERGED_ARCHIVE.stat().st_size != expected_archive_size:
    print("Creando el ZIP combinado...")
    temporary_archive = DOWNLOAD_DIR / "mp3.zip.tmp"
    with open(temporary_archive, "wb") as output_file:
        for part in PARTS:
            with open(DOWNLOAD_DIR / part, "rb") as input_file:
                shutil.copyfileobj(input_file, output_file)
    subprocess.run(["unzip", "-tq", str(temporary_archive)], check=True)
    temporary_archive.replace(MERGED_ARCHIVE)
else:
    print("El ZIP combinado ya existe; se reutiliza.")

EXTRACTION_MARKER = AUDIO_DIR / ".extraction_complete"
existing_mp3_count = sum(1 for _ in AUDIO_DIR.rglob("*.mp3"))
extraction_is_complete = False

if EXTRACTION_MARKER.exists():
    try:
        previous_mp3_count = int(EXTRACTION_MARKER.read_text().strip())
        extraction_is_complete = (
            existing_mp3_count == previous_mp3_count and existing_mp3_count > 0
        )
    except (ValueError, OSError):
        extraction_is_complete = False

if extraction_is_complete:
    print(f"Extracción completa ({existing_mp3_count:,} MP3); se reutiliza.")
else:
    print("Extrayendo los audios...")
    subprocess.run(
        ["unzip", "-q", "-o", str(MERGED_ARCHIVE), "-d", str(AUDIO_DIR)],
        check=True
    )
    extracted_mp3_count = sum(1 for _ in AUDIO_DIR.rglob("*.mp3"))
    if extracted_mp3_count == 0:
        raise RuntimeError("No se encontraron archivos MP3 tras la extracción.")
    EXTRACTION_MARKER.write_text(str(extracted_mp3_count))
    print(f"Extracción terminada: {extracted_mp3_count:,} MP3.")

print("Carpeta de audios:", AUDIO_DIR)

El ZIP combinado ya existe; se reutiliza.
Extracción completa (25,863 MP3); se reutiliza.
Carpeta de audios: /content/drive/MyDrive/MLSM/Lab 3/data/audio


## 4. Localización y comprobación de los audios

Se comprueba que los archivos MP3 pueden leerse y que duran al menos un segundo. El resultado se guarda en una caché persistente: en sesiones posteriores solo se revisan las firmas de los archivos, sin volver a decodificarlos para comprobar su duración.

In [21]:
all_audio_paths = sorted(AUDIO_DIR.rglob("*.mp3"))
if not all_audio_paths:
    raise RuntimeError("No se han encontrado archivos MP3.")

PREPARED_DIR = DATA_ROOT / "prepared_v3"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_CACHE = PREPARED_DIR / "audio_validation_v3.json"

current_manifest = []
for path in all_audio_paths:
    stat = path.stat()
    current_manifest.append({
        "path": str(path.relative_to(AUDIO_DIR)),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns
    })

cache_valid = False
valid_audio_paths = []

if VALIDATION_CACHE.exists():
    try:
        cached = json.loads(VALIDATION_CACHE.read_text(encoding="utf-8"))
        cache_valid = cached.get("manifest") == current_manifest
        if cache_valid:
            valid_audio_paths = [
                AUDIO_DIR / p for p in cached["valid_audio_paths"]
            ]
            cache_valid = all(p.is_file() for p in valid_audio_paths)
    except (json.JSONDecodeError, KeyError, TypeError, OSError):
        cache_valid = False

if cache_valid:
    print("Caché de validación reutilizada.")
else:
    print("Validando los MP3 por primera vez o porque el dataset ha cambiado...")
    valid_audio_paths = []
    for path in tqdm(all_audio_paths, desc="Comprobando MP3"):
        try:
            duration = librosa.get_duration(path=str(path))
            if duration >= SEGMENT_SECONDS:
                valid_audio_paths.append(path)
        except Exception:
            continue

    if not valid_audio_paths:
        raise RuntimeError("No hay audios válidos de al menos un segundo.")

    cache_data = {
        "manifest": current_manifest,
        "valid_audio_paths": [
            str(p.relative_to(AUDIO_DIR)) for p in valid_audio_paths
        ]
    }
    VALIDATION_CACHE.write_text(
        json.dumps(cache_data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("Caché de validación guardada.")

print(f"MP3 encontrados: {len(all_audio_paths):,}")
print(f"Audios válidos: {len(valid_audio_paths):,}")
print(f"Audios omitidos: {len(all_audio_paths) - len(valid_audio_paths):,}")

Caché de validación reutilizada.
MP3 encontrados: 25,863
Audios válidos: 25,860
Audios omitidos: 3


## 5. Separación en entrenamiento y validación

Separamos las canciones antes de recortar los segmentos. Así, ningún fragmento de una misma canción puede aparecer simultáneamente en entrenamiento y validación.

La división es 80/20, como en el notebook de referencia.

In [22]:
train_paths, val_paths = train_test_split(
    valid_audio_paths,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

print(f"Audios de entrenamiento: {len(train_paths):,}")
print(f"Audios de validación: {len(val_paths):,}")

Audios de entrenamiento: 20,688
Audios de validación: 5,172


## 6. Preparación de segmentos de audio

Los audios de MTAT son más largos que los ejemplos de clase. Aplicamos la estrategia de recorte indicada en el enunciado: usamos el **primer segundo** de cada canción, igual que el preprocesamiento del notebook docente.

El audio se convierte a mono y 16 kHz, se rellena con ceros si fuera necesario y se normaliza para que el máximo absoluto sea 1.

Los segmentos se guardan como matrices `float32`. La caché incluye las rutas y firmas de los archivos, el split y los parámetros de preprocesamiento. Si cambias el número de épocas, el learning rate o la arquitectura, los MP3 preparados no se vuelven a procesar. Solo se regeneran si cambian los datos o el preprocesamiento.

In [23]:
CACHE_DIR = PREPARED_DIR
TRAIN_CACHE = CACHE_DIR / "train_audio_v3.npy"
VAL_CACHE = CACHE_DIR / "val_audio_v3.npy"
PREPROCESSING_CACHE = CACHE_DIR / "preprocessing_v3.json"

preprocessing_signature = {
    "sample_rate": SAMPLE_RATE,
    "segment_seconds": SEGMENT_SECONDS,
    "input_length": INPUT_LENGTH,
    "crop": "first_segment",
    "normalization": "peak_1",
    "train_paths": [str(p.relative_to(AUDIO_DIR)) for p in train_paths],
    "val_paths": [str(p.relative_to(AUDIO_DIR)) for p in val_paths],
    "manifest": current_manifest
}

def prepare_audio_matrix(audio_paths, output_path):
    output_path = Path(output_path)
    temporary_path = output_path.with_suffix(".tmp.npy")

    matrix = np.lib.format.open_memmap(
        temporary_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(audio_paths), INPUT_LENGTH)
    )

    for index, path in enumerate(tqdm(audio_paths, desc=f"Preparando {output_path.name}")):
        audio, _ = librosa.load(
            path=str(path),
            sr=SAMPLE_RATE,
            mono=True
        )

        if len(audio) < INPUT_LENGTH:
            audio = np.pad(audio, (0, INPUT_LENGTH - len(audio)))

        # Mismo criterio que el notebook docente: primeros 16 000 samples.
        segment = audio[:INPUT_LENGTH].astype(np.float32)

        peak = np.max(np.abs(segment))
        if peak > 0:
            segment = segment / peak

        matrix[index] = segment

    matrix.flush()
    del matrix
    temporary_path.replace(output_path)
    print(f"Guardado: {output_path}")

def audio_cache_is_valid(cache_path, expected_rows):
    """Comprueba que la matriz .npy existe, se puede abrir y tiene forma y tipo correctos."""
    try:
        cached_audio = np.load(cache_path, mmap_mode="r")
        is_valid = (
            cached_audio.shape == (expected_rows, INPUT_LENGTH)
            and cached_audio.dtype == np.float32
        )
        del cached_audio
        return is_valid
    except (OSError, ValueError):
        return False


cache_matches = False
if PREPROCESSING_CACHE.exists() and TRAIN_CACHE.exists() and VAL_CACHE.exists():
    try:
        previous_signature = json.loads(
            PREPROCESSING_CACHE.read_text(encoding="utf-8")
        )
        cache_matches = (
            previous_signature == preprocessing_signature
            and audio_cache_is_valid(TRAIN_CACHE, len(train_paths))
            and audio_cache_is_valid(VAL_CACHE, len(val_paths))
        )
    except (json.JSONDecodeError, OSError, TypeError):
        cache_matches = False

if cache_matches:
    print("Los segmentos preparados coinciden con los datos y la configuración. Se reutilizan.")
else:
    print("No hay una caché válida. Se preparan los segmentos de audio.")
    prepare_audio_matrix(train_paths, TRAIN_CACHE)
    prepare_audio_matrix(val_paths, VAL_CACHE)
    PREPROCESSING_CACHE.write_text(
        json.dumps(preprocessing_signature, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("Metadatos de preprocesamiento guardados.")

Los segmentos preparados coinciden con los datos y la configuración. Se reutilizan.


## 7. Dataset y DataLoaders

El Dataset lee los segmentos directamente desde las matrices guardadas. No vuelve a abrir ni decodificar los MP3 durante el entrenamiento.

Creamos un DataLoader nuevo para cada modelo con el mismo generador aleatorio, de forma que ambos reciben los ejemplos en el mismo orden.

In [24]:
class PreparedAudioDataset(Dataset):
    def __init__(self, audio_file):
        self.audio_file = Path(audio_file)
        self.audio_matrix = np.load(self.audio_file, mmap_mode="r")

    def __len__(self):
        return self.audio_matrix.shape[0]

    def __getitem__(self, index):
        audio = np.array(self.audio_matrix[index], dtype=np.float32, copy=True)
        return torch.from_numpy(audio)

train_dataset = PreparedAudioDataset(TRAIN_CACHE)
val_dataset = PreparedAudioDataset(VAL_CACHE)

def make_train_loader(seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        generator=generator
    )

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Ejemplos de entrenamiento:", len(train_dataset))
print("Ejemplos de validación:", len(val_dataset))
print("Forma de un lote:", next(iter(make_train_loader(SEED))).shape)

Ejemplos de entrenamiento: 20688
Ejemplos de validación: 5172
Forma de un lote: torch.Size([32, 16000])


## 8. Arquitectura baseline: referencia docente

El baseline reproduce la arquitectura `AudioAutoencoder` del notebook de referencia y conserva sus parámetros principales:

- Entrada: 16 000 muestras.
- Tres bloques convolucionales con 8, 16 y 32 canales.
- Batch Normalization y PReLU.
- Tres MaxPool1d con índices y tres MaxUnpool1d que reutilizan esos índices.
- Bottleneck de 512 valores.
- Adam con learning rate 0.001 y MSE.

La tasa de compresión se mantiene en 16 000/512 = 31,25:1.

In [25]:
class AudioAutoencoder(nn.Module):
    def __init__(
        self,
        input_length=16000,
        bottleneck_size=512,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=2,
        improved=False
    ):
        super().__init__()
        assert kernel_size % 2 == 1
        self.improved = improved
        padding = kernel_size // 2

        # Encoder: arquitectura original del notebook de referencia.
        self.enc_conv1 = nn.Conv1d(1, channels, kernel_size, padding=padding)
        self.enc_norm1 = nn.BatchNorm1d(channels)
        self.enc_act1 = nn.PReLU()
        self.pool1 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv2 = nn.Conv1d(channels, channels * channel_mult, kernel_size, padding=padding)
        self.enc_norm2 = nn.BatchNorm1d(channels * channel_mult)
        self.enc_act2 = nn.PReLU()
        self.pool2 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv3 = nn.Conv1d(channels * channel_mult, channels * channel_mult ** 2, kernel_size, padding=padding)
        self.enc_norm3 = nn.BatchNorm1d(channels * channel_mult ** 2)
        self.enc_act3 = nn.PReLU()
        self.pool3 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        enc_out_len = input_length // downsample ** 3 * channels * channel_mult ** 2
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)

        print(f"Compression rate: {input_length / bottleneck_size:.2f}X")

        # Capas adicionales SOLO en el modelo mejorado.
        if improved:
            self.enc_extra1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
            self.enc_extra2 = nn.Conv1d(channels * channel_mult, channels * channel_mult, kernel_size, padding=padding)
            self.enc_extra3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult ** 2, kernel_size, padding=padding)

        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpool3 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult, kernel_size, padding=padding)
        self.dec_act3 = nn.PReLU()

        self.unpool2 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv2 = nn.Conv1d(channels * channel_mult, channels, kernel_size, padding=padding)
        self.dec_act2 = nn.PReLU()

        self.unpool1 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv1 = nn.Conv1d(channels, 1, kernel_size, padding=padding)
        self.dec_tanh = nn.Tanh()

        if improved:
            self.dec_extra3 = nn.Conv1d(channels * channel_mult, channels * channel_mult, kernel_size, padding=padding)
            self.dec_extra2 = nn.Conv1d(channels, channels, kernel_size, padding=padding)

    def encoder(self, x):
        sizes, indices = [], []

        x = self.enc_act1(self.enc_norm1(self.enc_conv1(x)))
        if self.improved:
            x = self.enc_act1(self.enc_extra1(x))
        sizes.append(x.size())
        x, idx1 = self.pool1(x)
        indices.append(idx1)

        x = self.enc_act2(self.enc_norm2(self.enc_conv2(x)))
        if self.improved:
            x = self.enc_act2(self.enc_extra2(x))
        sizes.append(x.size())
        x, idx2 = self.pool2(x)
        indices.append(idx2)

        x = self.enc_act3(self.enc_norm3(self.enc_conv3(x)))
        if self.improved:
            x = self.enc_act3(self.enc_extra3(x))
        sizes.append(x.size())
        x, idx3 = self.pool3(x)
        indices.append(idx3)

        x = x.flatten(start_dim=1)
        x = self.enc_fc(x)
        return x, indices, sizes

    def decoder(self, z, indices, sizes):
        idx1, idx2, idx3 = indices
        size1, size2, size3 = sizes

        x = self.dec_fc(z)
        x = x.view(size3[0], size3[1], -1)

        x = self.unpool3(x, idx3, output_size=size3)
        x = self.dec_act3(self.dec_deconv3(x))
        if self.improved:
            x = self.dec_act3(self.dec_extra3(x))

        x = self.unpool2(x, idx2, output_size=size2)
        x = self.dec_act2(self.dec_deconv2(x))
        if self.improved:
            x = self.dec_act2(self.dec_extra2(x))

        x = self.unpool1(x, idx1, output_size=size1)
        x = self.dec_tanh(self.dec_deconv1(x))
        return x

    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        reconstruction = self.decoder(z, indices, sizes)
        return reconstruction, z

# Comprobamos dimensiones y que ambos conservan el mismo bottleneck.
for improved in (False, True):
    check_model = AudioAutoencoder(
        input_length=INPUT_LENGTH,
        bottleneck_size=BOTTLENECK_SIZE,
        downsample=2,
        improved=improved
    ).to(DEVICE)
    with torch.no_grad():
        test = torch.zeros(2, 1, INPUT_LENGTH, device=DEVICE)
        reconstructed, latent = check_model(test)
    print("Improved:", improved, "| salida:", reconstructed.shape, "| bottleneck:", latent.shape)
    del check_model, test, reconstructed, latent

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Compression rate: 31.25X
Improved: False | salida: torch.Size([2, 1, 16000]) | bottleneck: torch.Size([2, 512])
Compression rate: 31.25X
Improved: True | salida: torch.Size([2, 1, 16000]) | bottleneck: torch.Size([2, 512])


## 9. Pérdidas

### Baseline: MSE

El baseline se entrena con el error cuadrático medio entre el audio original y su reconstrucción.

### Modelo mejorado: pérdida perceptual multiescala STFT

Se utiliza la función `multi_scale_stft_loss` del notebook docente, con tamaños FFT de 512, 1024 y 2048. Esta pérdida compara la magnitud de los espectros a distintas resoluciones.

La pérdida STFT se utiliza para entrenar el modelo mejorado. Para responder al criterio del enunciado, **la evaluación y la comparación de ambos modelos se hacen mediante MSE de validación**.

In [26]:
mse_criterion = nn.MSELoss()

def multi_scale_stft_loss(x, y, scales=(512, 1024, 2048), hop_ratio=0.25):
    """Pérdida perceptual multiescala del notebook docente."""
    loss = 0.0
    x = x.squeeze(1)
    y = y.squeeze(1)

    for n_fft in scales:
        hop = int(n_fft * hop_ratio)
        window = torch.hann_window(n_fft, device=x.device)
        X = torch.stft(
            x, n_fft=n_fft, hop_length=hop,
            window=window, return_complex=True
        )
        Y = torch.stft(
            y, n_fft=n_fft, hop_length=hop,
            window=window, return_complex=True
        )
        loss = loss + torch.mean(torch.abs(X.abs() - Y.abs()))

    return loss / len(scales)

## 10. Entrenamiento y evaluación

Se sigue el esquema de entrenamiento del notebook docente: Adam, 20 épocas y learning rate 0.001. No se introduce early stopping ni otra estrategia de optimización.

En cada época se calcula la pérdida de entrenamiento correspondiente al experimento. Además, se calcula el MSE de validación para ambos modelos y se guardan los pesos de la época con menor MSE.

In [27]:
def evaluate_mse(model, loader):
    model.eval()
    total_squared_error = 0.0
    total_elements = 0

    with torch.no_grad():
        for batch in loader:
            x = batch.unsqueeze(1).to(DEVICE)
            reconstruction, _ = model(x)
            total_squared_error += torch.sum((reconstruction - x) ** 2).item()
            total_elements += x.numel()

    return total_squared_error / total_elements


def train_model(model, train_loader, val_loader, improved=False, name="Baseline"):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {"train_loss": [], "val_mse": []}
    best_val_mse = float("inf")
    best_epoch = 0
    best_state = None
    checkpoint_path = RESULTS_DIR / f"{name.lower()}_best_v3.pt"

    for epoch in range(NUM_EPOCHS):
        model.train()
        train_losses = []

        for batch in tqdm(train_loader, desc=f"{name} | época {epoch + 1}/{NUM_EPOCHS}"):
            x = batch.unsqueeze(1).to(DEVICE)
            optimizer.zero_grad()

            reconstruction, _ = model(x)
            if improved:
                loss = multi_scale_stft_loss(x, reconstruction)
            else:
                loss = mse_criterion(reconstruction, x)

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_mse = evaluate_mse(model, val_loader)
        history["train_loss"].append(train_loss)
        history["val_mse"].append(val_mse)

        print(
            f"{name} | época {epoch + 1}/{NUM_EPOCHS} "
            f"| train loss: {train_loss:.6f} | validation MSE: {val_mse:.6f}"
        )

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_epoch = epoch + 1
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            torch.save(best_state, checkpoint_path)

    if best_state is None:
        raise RuntimeError(f"No se ha guardado ningún estado para {name}.")

    model.load_state_dict(best_state)
    print(f"Mejor MSE de validación ({name}): {best_val_mse:.6f}, época {best_epoch}")
    print("Checkpoint:", checkpoint_path)
    return model, history, best_val_mse, best_epoch

## 11. Entrenamiento del baseline

Se restablece la semilla antes de inicializar el modelo y se crea el DataLoader con el generador fijado. El baseline utiliza exactamente la arquitectura y la pérdida MSE del notebook docente.

In [29]:

# Entrenamiento o carga del baseline

set_seed(SEED)

baseline_checkpoint = RESULTS_DIR / "baseline_best_v3.pt"
baseline_history_file = RESULTS_DIR / "baseline_history_v3.csv"
baseline_summary_file = RESULTS_DIR / "baseline_summary_v3.csv"
baseline_config_file = RESULTS_DIR / "baseline_config_v3.json"

# Identificador de la arquitectura.
# Si modificas la arquitectura del baseline, incrementa esta versión.
BASELINE_ARCHITECTURE_VERSION = 1

# Configuración que determina el resultado del entrenamiento.
import hashlib

preprocessing_hash = hashlib.sha256(
    PREPROCESSING_CACHE.read_bytes()
).hexdigest()

baseline_config = {
    "architecture_version": BASELINE_ARCHITECTURE_VERSION,
    "input_length": INPUT_LENGTH,
    "bottleneck_size": BOTTLENECK_SIZE,
    "downsample": 2,
    "improved": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
    "loss": "MSE",
    "preprocessing_hash": preprocessing_hash
}

# Comprobar si existe un entrenamiento reutilizable.
baseline_cache_valid = False

required_files = [
    baseline_checkpoint,
    baseline_history_file,
    baseline_summary_file
]

if all(path.exists() for path in required_files):
    try:
        if baseline_config_file.exists():
            previous_config = json.loads(
                baseline_config_file.read_text(encoding="utf-8")
            )
            baseline_cache_valid = previous_config == baseline_config
        else:
            # Adopción de resultados anteriores de V3:
            # se comprueba que los tres archivos puedan leerse.
            # Se asume que proceden de la configuración actual.
            baseline_cache_valid = True

        if baseline_cache_valid:
            baseline_history_df = pd.read_csv(baseline_history_file)
            baseline_summary_df = pd.read_csv(baseline_summary_file)

            baseline_cache_valid = (
                len(baseline_history_df) == NUM_EPOCHS
                and not baseline_summary_df.empty
            )

    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        baseline_cache_valid = False

# Crear la arquitectura y cargar o entrenar.
baseline_model = AudioAutoencoder(
    input_length=INPUT_LENGTH,
    bottleneck_size=BOTTLENECK_SIZE,
    downsample=2,
    improved=False
).to(DEVICE)

if baseline_cache_valid:
    print("Baseline ya entrenado. Se reutilizan los resultados.")

    state_dict = torch.load(
        baseline_checkpoint,
        map_location=DEVICE,
        weights_only=True
    )
    baseline_model.load_state_dict(state_dict)

    baseline_history = {
        "train_loss": baseline_history_df["train_loss"].tolist(),
        "val_mse": baseline_history_df["val_mse"].tolist()
    }

    baseline_best_mse = float(
        baseline_summary_df.iloc[0]["best_validation_mse"]
    )
    baseline_best_epoch = int(
        baseline_summary_df.iloc[0]["best_epoch"]
    )
    baseline_parameters = sum(
        p.numel() for p in baseline_model.parameters()
        if p.requires_grad
    )

    # Guardar la configuración si se están adoptando
    # resultados de una ejecución anterior sin metadatos.
    if not baseline_config_file.exists():
        baseline_config_file.write_text(
            json.dumps(baseline_config, indent=2),
            encoding="utf-8"
        )

else:
    print("No existe un baseline válido. Se inicia el entrenamiento.")

    baseline_train_loader = make_train_loader(SEED)

    baseline_model, baseline_history, baseline_best_mse, baseline_best_epoch = train_model(
        baseline_model,
        baseline_train_loader,
        val_loader,
        improved=False,
        name="Baseline"
    )

    baseline_parameters = sum(
        p.numel() for p in baseline_model.parameters()
        if p.requires_grad
    )

    pd.DataFrame(baseline_history).to_csv(
        baseline_history_file,
        index_label="epoch"
    )

    pd.DataFrame([{
        "best_validation_mse": baseline_best_mse,
        "best_epoch": baseline_best_epoch,
        "trainable_parameters": baseline_parameters
    }]).to_csv(baseline_summary_file, index=False)

    baseline_config_file.write_text(
        json.dumps(baseline_config, indent=2),
        encoding="utf-8"
    )

    print("Baseline entrenado y resultados guardados.")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Compression rate: 31.25X
Baseline ya entrenado. Se reutilizan los resultados.


## 12. Entrenamiento del modelo mejorado

Respecto al baseline, solo se introducen los cambios contemplados en el enunciado:

1. Se añaden convoluciones adicionales en los bloques del encoder y del decoder.
2. Se entrena con la pérdida perceptual multiescala STFT del notebook docente.

Se conservan la longitud de entrada, los tres pasos de pooling, el tamaño del bottleneck, el conjunto de datos, los segmentos, el optimizador y el número de épocas. El resultado se selecciona según el menor MSE de validación.

In [30]:
set_seed(SEED)
improved_train_loader = make_train_loader(SEED)

improved_model = AudioAutoencoder(
    input_length=INPUT_LENGTH,
    bottleneck_size=BOTTLENECK_SIZE,
    downsample=2,
    improved=True
)

improved_model, improved_history, improved_best_mse, improved_best_epoch = train_model(
    improved_model,
    improved_train_loader,
    val_loader,
    improved=True,
    name="Improved"
)

improved_parameters = sum(p.numel() for p in improved_model.parameters() if p.requires_grad)
pd.DataFrame(improved_history).to_csv(
    RESULTS_DIR / "improved_history_v3.csv", index_label="epoch"
)
pd.DataFrame([{
    "best_validation_mse": improved_best_mse,
    "best_epoch": improved_best_epoch,
    "trainable_parameters": improved_parameters
}]).to_csv(RESULTS_DIR / "improved_summary_v3.csv", index=False)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Compression rate: 31.25X


Improved | época 1/20:   0%|          | 0/647 [00:00<?, ?it/s]

Improved | época 1/20 | train loss: 0.837790 | validation MSE: 0.135715


Improved | época 2/20:   0%|          | 0/647 [00:00<?, ?it/s]

Improved | época 2/20 | train loss: 0.610827 | validation MSE: 0.168308


Improved | época 3/20:   0%|          | 0/647 [00:00<?, ?it/s]

... (31 outputs left out)

Improved | época 19/20:   0%|          | 0/647 [00:00<?, ?it/s]

Improved | época 19/20 | train loss: 0.392786 | validation MSE: 0.209981


Improved | época 20/20:   0%|          | 0/647 [00:00<?, ?it/s]

Improved | época 20/20 | train loss: 0.390398 | validation MSE: 0.205691
Mejor MSE de validación (Improved): 0.135715, época 1
Checkpoint: /content/drive/MyDrive/MLSM/Lab 3/results_v3/improved_best_v3.pt


## 13. Comparación de resultados

La métrica de comparación es el MSE de validación. Un valor menor indica una reconstrucción más próxima a la señal original según el error cuadrático medio.

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "Best validation MSE": [baseline_best_mse, improved_best_mse],
    "Best epoch": [baseline_best_epoch, improved_best_epoch],
    "Trainable parameters": [baseline_parameters, improved_parameters]
})
display(results)

relative_improvement = (
    (baseline_best_mse - improved_best_mse) / baseline_best_mse
) * 100

print(f"Mejora relativa del MSE: {relative_improvement:.2f}%")
if improved_best_mse < baseline_best_mse:
    print("El modelo mejorado obtiene menor MSE de validación.")
elif improved_best_mse > baseline_best_mse:
    print("El baseline obtiene menor MSE de validación.")
else:
    print("Ambos modelos obtienen el mismo MSE de validación.")

results.to_csv(RESULTS_DIR / "model_comparison_v3.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(baseline_history["train_loss"], label="Baseline")
axes[0].plot(improved_history["train_loss"], label="Improved")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(baseline_history["val_mse"], label="Baseline")
axes[1].plot(improved_history["val_mse"], label="Improved")
axes[1].set_title("Validation MSE")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("MSE")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves_v3.png", dpi=200, bbox_inches="tight")
plt.show()

## 14. Escucha y visualización de reconstrucciones

Mostramos un ejemplo del conjunto de validación para escuchar y comparar la señal original y las dos reconstrucciones.

In [ ]:
batch = next(iter(val_loader))
example = batch[0].unsqueeze(0).unsqueeze(1).to(DEVICE)

baseline_model.eval()
improved_model.eval()
with torch.no_grad():
    baseline_reconstruction, _ = baseline_model(example)
    improved_reconstruction, _ = improved_model(example)

original_audio = example[0, 0].cpu().numpy()
baseline_audio = baseline_reconstruction[0, 0].cpu().numpy()
improved_audio = improved_reconstruction[0, 0].cpu().numpy()

display(Audio(original_audio, rate=SAMPLE_RATE))
display(Audio(baseline_audio, rate=SAMPLE_RATE))
display(Audio(improved_audio, rate=SAMPLE_RATE))

time_axis = np.arange(INPUT_LENGTH) / SAMPLE_RATE
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(time_axis, original_audio)
axes[0].set_title("Audio original")
axes[1].plot(time_axis, baseline_audio)
axes[1].set_title("Reconstrucción baseline")
axes[2].plot(time_axis, improved_audio)
axes[2].set_title("Reconstrucción mejorada")
axes[2].set_xlabel("Tiempo (segundos)")
for ax in axes:
    ax.set_ylabel("Amplitud")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "audio_reconstruction_comparison_v3.png", dpi=200, bbox_inches="tight")
plt.show()

import soundfile as sf
sf.write(RESULTS_DIR / "validation_original_v3.wav", original_audio, SAMPLE_RATE)
sf.write(RESULTS_DIR / "validation_baseline_reconstruction_v3.wav", baseline_audio, SAMPLE_RATE)
sf.write(RESULTS_DIR / "validation_improved_reconstruction_v3.wav", improved_audio, SAMPLE_RATE)

## 15. Conclusiones

Completa esta sección después de ejecutar el notebook con los resultados obtenidos.

1. ¿Qué modelo obtiene menor MSE de validación?
2. ¿En qué época obtiene cada modelo su mejor resultado?
3. ¿Qué efecto tienen las capas adicionales y la pérdida perceptual?
4. ¿La escucha de las reconstrucciones concuerda con el MSE?

## Reutilización de datos

Los MP3 descargados y extraídos permanecen en `MyDrive/MLSM/Lab 3/data`. Los segmentos procesados de V3 se guardan en `data/prepared_v3`.

Si modificas parámetros del modelo o del entrenamiento, puedes volver a ejecutar el notebook sin volver a decodificar los MP3. La caché se invalida únicamente cuando cambian los archivos del dataset, el split o los parámetros de preprocesamiento que determinan el contenido de los segmentos.

Los resultados de los modelos, historiales, tablas y gráficas se guardan en `MyDrive/MLSM/Lab 3/results_v3`.